In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing import TypedDict
import subprocess, textwrap
from openai import OpenAI
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]

In [ ]:
def extract_audio(state: State):
    # ffmpeg
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"], 
        "-filter:a",
        "atempo=2.0",
        "-y",
        output_file
    ]
    subprocess.run(command)
    return {
        "audio_file": output_file,
    }

def transcribe_audio(state: State):
    # use audio file
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="ko",
            prompt="창의성, 사고력, AI" # hint for Model
        )
        return {
            "transcription": transcription
        }

def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for idx, chunk in enumerate(textwrap.wrap(transcription, 200)):
        chunks.append({
            "id": idx+1,
            "chunk": chunk,
        })
    return [Send("summarize_chunk", chunk) for chunk in chunks]

def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {
        "summaries": [summary]
    }

    # print(f"Summarizing chunk id: {chunk_id}")
    # print(f"Chunk: {chunk[:100]}\n\n====\n\n")

In [3]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges("transcribe_audio", dispatch_summarizers , ["summarize_chunk"])
graph_builder.add_edge("summarize_chunk", END)

graph = graph_builder.compile()

graph.invoke({ "video_file": "asset/one_minute_video.mp4" })

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

Summarizing chunk id: 1Summarizing chunk id: 2
Chunk: 기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각

====



Chunk: 딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향

====




{'video_file': 'asset/one_minute_video.mp4',
 'audio_file': 'asset/one_minute_video.mp3',
 'transcription': '딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향을 미치는 과도적인 이유가 있더라고 미국의 MIT 미디어랩에서 사람들을 모아다가 SA를 작성하는 시험을 했거든 일단 팀을 세 개로 나눴는데 한 팀은 무조건 머리만 굴려서 SA를 쓰게 했고 다른 한 팀은 공부 같은 검색엔진 나머지 한 팀은 그냥 우리의 예상처럼 채 GPT가 된 LRM을 쓰게 했지 그리고 SA를 작성하는 동안만 우리가 어떻게 돌아가는지 확인하기 위해 뇌파를 확인했어 그래서 우리 머리로만 승부한 뒤 뇌의 연결성이 가장 유미했던 거야 아주 강력하고 분산된 네트워크로 온 뇌를 잘 굴린 거지 그리고 검색엔진을 쓴 그룹은 중간 정도 LRM을 쓴 그룹은 뇌의 연결성이 가장 약했어 우리를 제일 안 썼다는 거야 당연히 SA에서 자신의 주장 또한 제일 부족했고 시간이 조금만 지나도 본인이 대체 붙었는지 제대로 기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각에 유존한다면 우리의 사고력은 점차 태화한다는 거야 안 가서는 대대로 된 사고가 안 된다는 건데 형 이제 학교 공부는 버거워지고 시험 문제도 못 그리겠지 차라리 스스로 생각하고 이후에 AI의 도움을 받는 게 훨씬 좋아 이 편집은 뭐 옛날 연애하는 방법 AI랑 상관없어\n'}

In [4]:
transciption = "딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향을 미치는 과도적인 이유가 있더라고 미국의 MIT 미디어랩에서 사람들을 모아다가 SA를 작성하는 시험을 했거든 일단 팀을 세 개로 나눴는데 한 팀은 무조건 머리만 굴려서 SA를 쓰게 했고 다른 한 팀은 공부 같은 검색엔진 나머지 한 팀은 그냥 우리의 예상처럼 채 GPT가 된 LRM을 쓰게 했지 그리고 SA를 작성하는 농합비가 어떻게 돌아가는지 확인하기 위해 뇌파를 확인했어 그래서 우리 머리로만 승부한 뒤 뇌의 연결성이 가장 유미했던 거야 아주 강력하고 분산된 네트워크로 온 뇌를 잘 굴린 거지 그리고 검색엔진을 쓴 그룹은 중간 정도 LRM을 쓴 그룹은 뇌의 연결성이 가장 약했어 우리를 제일 안 썼다는 거야 당연히 SA에서 자신의 주장 또한 제일 부족했고 시간이 조금만 지나도 본인이 대체 붙었는지 제대로 기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각에 유존한다면 우리의 사고력은 점차 태화한다는 거야 안 가서는 대대로 된 사고가 안 된다는 건데 형 이제 학교 공부는 버거워지고 시험 문제도 못 그리겠지 차라리 스스로 생각하고 이후에 AI의 도움을 받는 게 훨씬 좋아 이 편집은 뭐 옛날 연애하는 방법 AI랑 상관없어\n"

import textwrap # 텍스트를 특정 너비에 맞게 맞춰줌

for idx, chunk in enumerate(textwrap.wrap(transciption, 500)):
    print(f"Chunk ID: {idx}")
    print(chunk)
    print("======")

Chunk ID: 0
딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향을 미치는 과도적인 이유가 있더라고 미국의 MIT 미디어랩에서 사람들을 모아다가 SA를 작성하는 시험을 했거든 일단 팀을 세 개로 나눴는데 한 팀은 무조건 머리만 굴려서 SA를 쓰게 했고 다른 한 팀은 공부 같은 검색엔진 나머지 한 팀은 그냥 우리의 예상처럼 채 GPT가 된 LRM을 쓰게 했지 그리고 SA를 작성하는 농합비가 어떻게 돌아가는지 확인하기 위해 뇌파를 확인했어 그래서 우리 머리로만 승부한 뒤 뇌의 연결성이 가장 유미했던 거야 아주 강력하고 분산된 네트워크로 온 뇌를 잘 굴린 거지 그리고 검색엔진을 쓴 그룹은 중간 정도 LRM을 쓴 그룹은 뇌의 연결성이 가장 약했어 우리를 제일 안 썼다는 거야 당연히 SA에서 자신의 주장 또한 제일 부족했고 시간이 조금만 지나도 본인이 대체 붙었는지 제대로
Chunk ID: 1
기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각에 유존한다면 우리의 사고력은 점차 태화한다는 거야 안 가서는 대대로 된 사고가 안 된다는 건데 형 이제 학교 공부는 버거워지고 시험 문제도 못 그리겠지 차라리 스스로 생각하고 이후에 AI의 도움을 받는 게 훨씬 좋아 이 편집은 뭐 옛날 연애하는 방법 AI랑 상관없어
